# Model 3: RAG (Retrieval-Augmented Generation)

**Retrieval:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (Semantische Suche)  
**Generation:** `Qwen/Qwen2.5-3B-Instruct` (Exzellentes Deutsch)


In [ ]:
# ZELLE 1: Pakete installieren
!pip install -q transformers torch accelerate pypdf sentence-transformers tqdm

In [ ]:
# ZELLE 2: GPU-Check
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Gerät: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ZELLE 3: Pfade
import os
DATA_DIR = '/kaggle/input/datasets/lucariggler/wu-steuerrecht'
DATASET_PATH = os.path.join(DATA_DIR, 'dataset_clean.csv')
OUTPUT_PATH = '/kaggle/working/model_3_output.csv'


In [ ]:
# ZELLE 4: PDF-Wissensbasis laden (überlappende Chunks)
from pypdf import PdfReader

def load_pdf_corpus(data_dir, words_per_chunk=80, overlap=20):
    corpus = []
    for filename in sorted(os.listdir(data_dir)):
        if not filename.endswith('.pdf'):
            continue
        print(f'Lese PDF: {filename}')
        try:
            reader = PdfReader(os.path.join(data_dir, filename))
            full_text = ' '.join(
                page.extract_text().replace('\n', ' ')
                for page in reader.pages if page.extract_text()
            )
            words = full_text.split()
            step = words_per_chunk - overlap
            for i in range(0, len(words), step):
                chunk = ' '.join(words[i:i + words_per_chunk])
                if len(chunk) > 80:
                    corpus.append(chunk)
        except Exception as e:
            print(f'Fehler: {e}')
    corpus = list(dict.fromkeys(corpus))
    print(f'{len(corpus)} Textabschnitte geladen.')
    return corpus

corpus = load_pdf_corpus(DATA_DIR)
print('Beispiel:', corpus[0][:200])

In [ ]:
# ZELLE 5: Semantische Embeddings berechnen (einmalig ~5 Min.)
from sentence_transformers import SentenceTransformer
import numpy as np
import time

print('Lade multilinguales Embedding-Modell...')
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print('Berechne Embeddings für alle PDF-Absätze (einmalig)...')
start = time.time()
corpus_embeddings = embedder.encode(
    corpus,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(f'Fertig in {(time.time()-start)/60:.1f} Minuten. Shape: {corpus_embeddings.shape}')

In [ ]:
# ZELLE 6: Generierungs-Modell laden
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
GEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.float16, device_map='auto')
model.eval()


In [ ]:
# ZELLE 7: RAG Inferenz
import pandas as pd
from tqdm import tqdm
import numpy as np
import torch

def retrieve_context(question, top_k=3):
    q_emb = embedder.encode([question], normalize_embeddings=True, convert_to_numpy=True)
    scores = np.dot(corpus_embeddings, q_emb.T).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    return ' | '.join([corpus[i] for i in top_idx])[:1000]

df = pd.read_csv(DATASET_PATH, encoding='utf-8-sig')
pd.DataFrame(columns=['id', 'answer']).to_csv(OUTPUT_PATH, index=False)

for _, row in tqdm(df.iterrows(), total=len(df)):
    question = str(row.get('prompt', ''))
    context = retrieve_context(question)

    messages = [
        {'role': 'system', 'content': 'Du bist ein Experte für österreichisches Steuerrecht. Beantworte die Frage NUR basierend auf dem bereitgestellten Kontext.'},
        {'role': 'user', 'content': f'Kontext:\n{context}\n\nFrage:\n{question}'}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to('cuda')
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.1, do_sample=True, repetition_penalty=1.2, no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    
    ans = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
    if len(ans) > 500: ans = ans[:497] + '...'
    pd.DataFrame([{'id': row['id'], 'answer': ans}]).to_csv(OUTPUT_PATH, mode='a', header=False, index=False)


In [ ]:
# ZELLE 8: Ergebnis prüfen
df_out = pd.read_csv(OUTPUT_PATH)
print(f'Zeilen: {len(df_out)}')
df_out.head(10)

In [ ]:
# ZELLE 9: Download
from IPython.display import FileLink
print('Download:')
FileLink(OUTPUT_PATH)